# 15 — Export 3D data for ImageJ/Fiji and ParaView

This notebook exports reconstructed SIMS volumes to:

- ImageJ/Fiji hyperstack TIFF
- ParaView `.vti`

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import pandas as pd

from pymagsims import SIMSVolume
from pymagsims.export import export_imagej_hyperstack, export_paraview_vti

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"
EXPORT_DIR = DATA / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SHAPE = (256, 256)
CHANNEL_BIN_FILE = DATA / "bins" / "channel_bins_from_3d_csv_calibration.csv"

## 1. Rebuild volume and element volumes

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(RAW_LAYER_DIR.glob("*Image_*.raw"), key=natural_sort_key)
selected_bins = pd.read_csv(CHANNEL_BIN_FILE)

volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

elements = ["Si", "Ti", "Mg", "Hf"]

element_volumes = {}

for element in elements:
    try:
        element_volumes[element] = volume.summed_by_element(selected_bins, element)
        print(element, element_volumes[element].sum())
    except ValueError as exc:
        print(f"Skipping {element}: {exc}")

arrays = {
    "Total": volume.get("Total"),
    **element_volumes,
}

print(arrays.keys())

## 2. Export ImageJ/Fiji hyperstack

In [ ]:
imagej_file = EXPORT_DIR / "ion_hyperstack.tif"

export_imagej_hyperstack(
    arrays,
    imagej_file,
    dtype="float32",
)

print(f"Exported ImageJ/Fiji hyperstack to: {imagej_file}")

## 3. Export ParaView VTI

`.vti` is recommended for dense voxel stacks.

In [ ]:
vti_file = EXPORT_DIR / "sims_volume.vti"

export_paraview_vti(
    arrays,
    vti_file,
    spacing=(1.0, 1.0, 1.0),
)

print(f"Exported ParaView VTI to: {vti_file}")

## 4. Export one VTI per element

In [ ]:
per_element_dir = EXPORT_DIR / "per_element"
per_element_dir.mkdir(parents=True, exist_ok=True)

for label, arr in arrays.items():
    out = per_element_dir / f"{label}.vti"
    export_paraview_vti(
        {label: arr},
        out,
        spacing=(1.0, 1.0, 1.0),
    )
    print(out)

## Fiji notes

Open:

```text
File → Open → ion_hyperstack.tif
```

If Fiji does not detect dimensions correctly:

```text
Image → Hyperstacks → Stack to Hyperstack
```

Use:

```text
channels = number of arrays
slices = number of layers
frames = 1
```

## ParaView notes

Open:

```text
File → Open → sims_volume.vti
```

Good workflows:

```text
Volume rendering
Contour → Surface
Clip → Box → Invert
Slice
Threshold
```